In [ ]:
import os, shutil, warnings, gc
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#  PATHS 

# Task 1: Nasdaq dataset
NASDAQ_RAW_DIR   = Path("./data/nasdaq/csv")          
NASDAQ_CLEAN_DIR = Path("./data/nasdaq/csv_clean")    

# Task 2: Vietnam stock dataset
VN_RAW_DIR       = Path("./data/vietnam/stock-historical-data")  
VN_VNINDEX_DIR   = Path("./data/vietnam/vnindex_only")   
VN_CLEAN_DIR     = Path("./data/vietnam/vnindex_clean")  

VN_DIVIDEND_DIR  = Path("./data/vietnam/dividend-history")
VN_INDUSTRY_DIR  = Path("./data/vietnam/industrial-analysis")
VN_FINRATIO_DIR  = Path("./data/vietnam/financial-ratio")


In [ ]:
_ALIASES = {
    "Date":   ["date","Date","time","Time","<DTYYYYMMDD>","<Date>", "TradingDate"],
    "Open":   ["open","Open","<Open>","<OPEN>"],
    "High":   ["high","High","<High>","<HIGH>"],
    "Low":    ["low","Low","<Low>","<LOW>"],
    "Close":  ["close","Close","<Close>","<CLOSE>"],
    "Volume": ["volume","Volume","<Volume>","<VOL>"],
}

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for canon, aliases in _ALIASES.items():
        for a in aliases:
            if a in df.columns and canon not in rename.values():
                rename[a] = canon
                break
    return df.rename(columns=rename)


def assess_csv(path: Path, required: List[str]) -> Tuple[bool, str]:
    """Return (keep, reason). Reason is '' when the file passes.

    Criteria
    --------
    1. File is readable and non-empty.
    2. All required columns are present.
    3. At least 1 000 rows (≈4 years of trading data).
    4. < 5 % missing values in any required numeric column.
    5. No run of > 20 consecutive NaN rows.
    6. Close price strictly positive everywhere.
    7. Median daily Volume > 0 in the last 252 rows.
    8. OHLC integrity ≥ 80 % of rows (High ≥ Low, etc.).
    """
    try:
        df = pd.read_csv(path)
    except Exception as e:
        return False, f"unreadable ({e.__class__.__name__})"

    df = standardize_columns(df)

    if df.empty:
        return False, "empty"

    missing = [c for c in required if c not in df.columns]
    if missing:
        return False, f"missing cols {missing}"

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    if len(df) < 1000:
        return False, f"only {len(df)} rows"

    for c in [x for x in required if x != "Date"]:
        mr = df[c].isna().mean()
        if mr > 0.05:
            return False, f"{c} {mr:.0%} missing"

    # consecutive NaN run
    for c in ["Open","High","Low","Close"]:
        if c not in df.columns: continue
        run = best = 0
        for v in df[c].isna().astype(int):
            run = run + 1 if v else 0
            best = max(best, run)
        if best > 20:
            return False, f"{c} gap {best} days"

    if (df["Close"].dropna() <= 0).any():
        return False, "non-positive close"

    if df["Volume"].dropna().tail(252).median() <= 0:
        return False, "zero recent volume"

    ohlc = df[["Open","High","Low","Close"]].dropna()
    sane = ((ohlc.High >= ohlc.Low) & (ohlc.High >= ohlc.Close) &
            (ohlc.High >= ohlc.Open) & (ohlc.Low  <= ohlc.Close) &
            (ohlc.Low  <= ohlc.Open)).mean()
    if sane < 0.80:
        return False, f"OHLC integrity {sane:.0%}"

    return True, ""


def filter_folder(src: Path, dst: Path, required_cols: List[str]) -> pd.DataFrame:
    """Scan every CSV in src; copy passing files to dst. Returns audit report."""
    src, dst = Path(src), Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    records = []
    files = sorted(src.glob("*.csv"))
    print(f"Scanning {len(files)} files …")
    for i, p in enumerate(files, 1):
        ok, reason = assess_csv(p, required_cols)
        if ok:
            shutil.copy2(p, dst / p.name)
        records.append({"file": p.name, "kept": ok, "reason": reason})
        if i % 300 == 0:
            print(f"  {i}/{len(files)} done")
    df = pd.DataFrame(records)
    kept = df.kept.sum()
    print(f"\nResult: {kept}/{len(df)} files kept ({kept/len(df):.1%})")
    return df


## Task 1: Filter Nasdaq dataset

In [ ]:
REQUIRED_NASDAQ = ["Date","Open","High","Low","Close","Volume"]

# Uncomment to run (scans all 1564 files, takes ~1–2 min):
nasdaq_report = filter_folder(NASDAQ_RAW_DIR, NASDAQ_CLEAN_DIR, REQUIRED_NASDAQ, max_kept=400)
print(nasdaq_report["reason"].value_counts().head(10))
print(f"Clean tickers: {len(list(NASDAQ_CLEAN_DIR.glob('*.csv')))}")


Scanning files to find 400 high-quality tickers...


/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarn

  Processed 100 files... (76/400 kept)


/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarn

  Processed 200 files... (153/400 kept)


...

/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarn

  Processed 400 files... (315/400 kept)


/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarn

  Processed 500 files... (393/400 kept)

Target reached: 400 files kept. Stopping early.

Result: 400 files kept.
reason
                      400
zero recent volume     16
Open 25% missing        5
Open 14% missing        4
Open 26% missing        3
Open 27% missing        2
Open 13% missing        2
OHLC integrity 80%      2
Open 17% missing        2
Open 10% missing        2
Name: count, dtype: int64
Clean tickers: 400


/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
/var/folders/f2/ng8gzz4x5pl14lc3rkmcz2140000gn/T/ipykernel_59311/3778433422.py:60: UserWarn

## Task 2: Filter VN dataset

In [ ]:
def copy_vnindex_only(src: Path, dst: Path) -> int:
    src, dst = Path(src), Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    total = kept = 0
    for p in sorted(src.glob("*.csv")):
        total += 1
        if "VNINDEX" in p.name.upper():
            shutil.copy2(p, dst / p.name)
            kept += 1
    print(f"Exchange filter: kept {kept}/{total} VNINDEX files")
    return kept

n_vn = copy_vnindex_only(VN_RAW_DIR, VN_VNINDEX_DIR)


Exchange filter: kept 416/1629 VNINDEX files


In [ ]:
REQUIRED_VN = ["Date","Open","High","Low","Close","Volume"]

vn_report = filter_folder(VN_VNINDEX_DIR, VN_CLEAN_DIR, REQUIRED_VN)
print(vn_report["reason"].value_counts().head(10))
print(f"Clean VNIndex tickers: {len(list(VN_CLEAN_DIR.glob('*.csv')))}")


Scanning 416 files …
  300/416 done

Result: 348/416 files kept (83.7%)
reason
                      348
zero recent volume      9
non-positive close      4
OHLC integrity 79%      3
OHLC integrity 78%      3
only 483 rows           2
OHLC integrity 66%      2
only 738 rows           2
only 781 rows           2
OHLC integrity 70%      1
Name: count, dtype: int64
Clean VNIndex tickers: 348
